### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="sdss_17",
    dataset_year="2022",
    domain_str="physics & astronomy",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/datasets/fedesoriano/stellar-classification-dataset-sdss17",
    download_description="""
We download the data from Kaggle.

mkdir -p local-data-warehouse/sdss_17/ && cd local-data-warehouse/sdss_17 && kaggle datasets download fedesoriano/stellar-classification-dataset-sdss17 && cd ../../ && unzip local-data-warehouse/sdss_17/stellar-classification-dataset-sdss17.zip -d local-data-warehouse/sdss_17 && rm local-data-warehouse/sdss_17/stellar-classification-dataset-sdss17.zip
""",
    # References
    academic_reference_bibtex=r"""@article{abazajian2009stars,
  title={The seventh data release of the Sloan Digital Sky Survey},
  author={Abazajian, Kevork N and Adelman-McCarthy, Jennifer K and Ag{\"u}eros, Marcel A and Allam, Sahar S and Prieto, Carlos Allende and An, Deokkeun and Anderson, Kurt SJ and Anderson, Scott F and Annis, James and Bahcall, Neta A and others},
  journal={The Astrophysical Journal Supplement Series},
  volume={182},
  number={2},
  pages={543--558},
  year={2009},
  publisher={The American Astronomical Society}
}
""",
    academic_reference_bibtex_key="abazajian2009stars",
    license="Public Domain",
    data_tags=["IID"],
    curation_comments="""
- We renamed the target feature.
- We dropped duplicates based on "obj_ID" to avoid target leakage from subgroups. 
- We dropped several (ID-like) meta-features that seem to be not part of the predictive task.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="ObjectType",
    problem_type="multiclass_classification",
    objective_metric_name="log_loss",
    stratify_on="ObjectType",
)

## Preprocessing

In [9]:
import pandas as pd

df = pd.read_csv(f"{dataset_mold.path}/star_classification.csv", header=0)

feature_names = [
    "obj_ID",
    "alpha",
    "delta",
    "u",
    "g",
    "r",
    "i",
    "z",
    "run_ID",
    "rerun_ID",
    "cam_col",
    "field_ID",
    "spec_obj_ID",
    "ObjectType",
    "redshift",
    "plate",
    "MJD",
    "fiber_ID"
]

df.columns = feature_names

cat_features = [
    "cam_col",
    "ObjectType",
    "plate",
    "fiber_ID",
]

df[cat_features] = df[cat_features].astype("category")

df = df.drop(
    columns=[
        "obj_ID",  # should not be predictive, also has duplicates?
        "spec_obj_ID",  # ID
        "run_ID",  # should not be predictive but might indicate confounding noise
        "rerun_ID",  # constant
        "field_ID",  # might indicate clusters/subgroups of data?
        "MJD",  # date of observation, should not be predictive?
    ]
)

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

In [10]:
# Use if needed to get see all cols of pandas dataframes
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)
df.head()

,alpha,delta,u,g,r,i,z,cam_col,ObjectType,redshift,plate,fiber_ID
0,16.956890,3.646130,23.33542,21.95143,20.48149,19.60300,19.13094,6,GALAXY,0.506237,4312,495
1,240.063240,6.134131,17.86033,16.79228,16.43001,16.30923,16.25873,1,STAR,0.000345,2175,348
2,30.887222,1.188710,18.18911,16.89469,16.42161,16.24627,16.18549,1,STAR,0.000004,7332,943
3,247.594401,10.887780,24.99961,21.71203,21.47148,21.30532,21.29109,1,STAR,-0.000291,4066,326
4,18.896451,-5.261330,23.76648,21.79737,20.69543,20.23403,19.97464,3,STAR,-0.000136,7914,363


## Data Checks

In [11]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 100,000
Columns: 12
Use sampling: False (sample size: 100,000)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['alpha', 'delta', 'redshift', 'u', 'g', 'i', 'z', 'r', 'plate', 'fiber_ID']
Rows remaining as candidates after top-10 filter: 0 (of 100,000)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [12]:
# Sample Rows
df_head

,alpha,delta,u,g,r,i,z,cam_col,ObjectType,redshift,plate,fiber_ID
0,16.956890,3.646130,23.33542,21.95143,20.48149,19.60300,19.13094,6,GALAXY,0.506237,4312,495
1,240.063240,6.134131,17.86033,16.79228,16.43001,16.30923,16.25873,1,STAR,0.000345,2175,348
2,30.887222,1.188710,18.18911,16.89469,16.42161,16.24627,16.18549,1,STAR,0.000004,7332,943
3,247.594401,10.887780,24.99961,21.71203,21.47148,21.30532,21.29109,1,STAR,-0.000291,4066,326
4,18.896451,-5.261330,23.76648,21.79737,20.69543,20.23403,19.97464,3,STAR,-0.000136,7914,363


In [14]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,cam_col,category,0.0,0.0,6.0,"4, 3, 5, 2, 1, 6"
1,ObjectType,category,0.0,0.0,3.0,"GALAXY, STAR, QSO"
2,plate,category,0.0,0.0,6284.0,"6301, 7699, 7407, 7147, 6516, 5185, 7697, 7450, 10431, 7701"
3,fiber_ID,category,0.0,0.0,1000.0,"637, 105, 597, 321, 611, 621, 564, 563, 409, 571"
4,alpha,float64,0.0,0.0,99999.0,"34.7496, 16.9569, 182.5288, 157.7958, 317.2748, 50.5426, 218.4061, 187.4628, 359.4065, 239.9886"
5,delta,float64,0.0,0.0,99999.0,"-0.6019, 3.6461, 29.5751, 37.5342, 0.1032, -0.466, 19.0037, -1.281, -7.2236, 33.2332"
6,u,float64,0.0,0.0,93748.0,"24.6347, 24.6347, 24.6347, 24.6346, 24.6347, 24.6346, 22.0871, 20.5925, 23.4334, 24.6347"
7,g,float64,0.0,0.0,92651.0,"25.1144, 25.1144, 25.1144, 22.0364, 22.5637, 21.0489, 20.4269, 22.4501, 21.1364, 22.4174"
8,r,float64,0.0,0.0,91901.0,"24.802, 24.802, 24.802, 20.7224, 20.7653, 20.7466, 20.2045, 20.5444, 20.7826, 20.2874"
9,i,float64,0.0,0.0,92019.0,"24.3618, 24.3618, 18.6331, 19.6762, 17.6259, 19.7496, 20.6126, 20.6777, 19.7921, 19.6016"


In [15]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
alpha,100000.0,177.629117,96.502241,0.005528,359.999810
delta,100000.0,24.135305,19.644665,-18.785328,83.000519
u,100000.0,21.980468,31.769291,-9999.000000,32.781390
g,100000.0,20.531387,31.750292,-9999.000000,31.602240
r,100000.0,19.645762,1.854760,9.822070,29.571860
i,100000.0,19.084854,1.757895,9.469903,32.141470
z,100000.0,18.668810,31.728152,-9999.000000,29.383740
redshift,100000.0,0.576661,0.730707,-0.009971,7.011245


In [16]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column     rank                      
ObjectType 1     GALAXY  59445  59.44
           2       STAR  21594  21.59
           3        QSO  18961  18.96
cam_col    1          4  19573  19.57
           2          3  18851  18.85
           3          5  18537  18.54
           4          2  17117  17.12
           5          1  13227  13.23
fiber_ID   1        637    159   0.16
           2        105    158   0.16
           3        597    158   0.16
           4        321    154   0.15
           5        611    154   0.15
plate      1       6301     98   0.10
           2       7699     97   0.10
           3       7407     96   0.10
           4       7147     95   0.10
           5       6516     94   0.09

In [17]:
# Target Distribution
target_df

,count,pct
ObjectType,,
GALAXY,59445,59.44
STAR,21594,21.59
QSO,18961,18.96


## Task Curation

In [18]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=3, n_splits=3, test_size=None


In [19]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [20]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to sdss_17/019d3e0d-4281-7b0f-b369-91d28045c658
019d3e0d-4281-7b0f-b369-91d28045c658
4fb65c2c071941f83d2c1b6a0b76cf4a36067b898826d26d5e66d1945b5e6e02
